# 09 — Functional Tools and itertools

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
# Provided for this lesson.
from collections.abc import Callable

from lesson import Exercises, ___, run

check = Exercises('intermediate/09', {1: '21ae3837f487dbfa', 2: 'c42f763781d11ca8', 3: 'b321e2674b06e349', 4: '5bc078685a521c5a', 5: 'dc572f87214d69dd', 6: '72d37575bce75d7f', 7: 'eeeb5a955c79f73c', 8: '0f2b8c2e7e6d7310', 9: 'e91b8e0d5b84c9a3', 10: '5ad30dcf9b50d83a', 11: '418802987d5281c1', 12: 'f4d41be131e6bcf3'}, practice='intermediate_02_iteration_and_abstractions', independent='totals_by_category')

## 1. lambda, sorted and key

- `lambda args: expression` is a small unnamed function, handy as a `key`.
- `sorted(rows, key=...)` sorts by what the key returns; `reverse=True` flips it.
- Sort by several fields with a tuple key; negate a number to sort that part descending: `key=lambda r: (r["team"], -r["salary"])`.

In [ ]:
rows = [{"name": "Ana", "team": "b", "salary": 50}, {"name": "Bo", "team": "a", "salary": 40}, {"name": "Cy", "team": "a", "salary": 60}]
for r in sorted(rows, key=lambda r: (r["team"], -r["salary"])):
    print(r["team"], r["salary"], r["name"])

**Exercise 1.** Sort words by length, shortest first.

In [ ]:
words = ["banana", "fig", "apple"]
result = sorted(words, key=___)
check(1, result)

**Exercise 2.** Sort numbers from largest to smallest.

In [ ]:
nums = [3, 9, 1]
result = sorted(nums, reverse=___)
check(2, result)

**Exercise 3.** Sort by team ascending, then salary descending. List the names in order.

In [ ]:
rows = [("Ana", "b", 50), ("Bo", "a", 40), ("Cy", "a", 60)]
ordered = sorted(rows, key=lambda r: (r[1], ___))
names = [r[0] for r in ordered]
check(3, names)

## 2. map, filter, reduce, partial

- `map(f, items)` applies `f` to each item; `filter(f, items)` keeps items where `f` is truthy. Both are lazy.
- `functools.reduce(f, items, start)` folds a list into one value.
- `functools.partial(f, x=...)` makes a new function with some arguments fixed.

In [ ]:
from functools import reduce, partial
print(list(map(str.upper, ["a", "b"])))
print(list(filter(lambda n: n % 2, range(6))))
print(reduce(lambda a, b: a * b, [1, 2, 3, 4], 1))
square = partial(pow, exp=2)
print(square(7))

**Exercise 4.** Convert every string to an int with `map`.

In [ ]:
texts = ["1", "20", "3"]
numbers = list(map(___, texts))
check(4, numbers)

**Exercise 5.** Keep only the words longer than 3 letters with `filter`.

In [ ]:
words = ["a", "tree", "cat", "house"]
long_words = list(filter(lambda w: ___, words))
check(5, long_words)

**Exercise 6.** Multiply all the numbers together with `reduce`.

In [ ]:
from functools import reduce
product = reduce(lambda a, b: a * b, [2, 3, 4], ___)
check(6, product)

**Exercise 7.** Make a `cube` function from `power` by fixing `exponent=3`.

In [ ]:
from functools import partial

def power(base, exponent):
    return base ** exponent

cube = partial(power, exponent=___)
check(7, cube(2))

## 3. lru_cache

- `@functools.lru_cache(maxsize=None)` remembers results by arguments, so repeated calls are instant.
- It turns slow recursive functions like Fibonacci from exponential to linear.
- Arguments must be hashable (no lists or dicts).

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80))
print(fib.cache_info().hits > 0)

**Exercise 8.** Finish the memoised Fibonacci: `fib(0) == 0`, `fib(1) == 1`.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return ___
    return fib(n - 1) + fib(n - 2)

check(8, fib(30))

**Exercise 9.** Can `lru_cache` cache a call with a list argument? Write `True` or `False`.

In [ ]:
answer = ___
check(9, answer)

## 4. product and groupby

- `itertools.product(a, b)` gives every pair, like nested loops with `a` outside.
- `itertools.groupby(items, key)` groups only neighbouring items, so sort by the same key first.
- Each group is `(key, iterator)`: consume the iterator before moving on.

In [ ]:
from itertools import product, groupby
print(list(product(["red", "blue"], ["S", "M"])))
rows = [("b", 2), ("a", 1), ("b", 3)]
rows.sort(key=lambda r: r[0])
print({k: sum(v for _, v in g) for k, g in groupby(rows, key=lambda r: r[0])})

**Exercise 10.** Every (colour, size) pair, colours outside.

In [ ]:
from itertools import product
pairs = list(product(["red", "blue"], ___))
check(10, pairs)

**Exercise 11.** `groupby` on unsorted letters. Predict the group keys in order.

In [ ]:
from itertools import groupby
keys = [k for k, _ in groupby("aabaa")]
predicted = ___
check(11, predicted)

**Exercise 12.** Total the amounts per category: sort first, then group.

In [ ]:
from itertools import groupby
rows = [("food", 5), ("rent", 100), ("food", 7)]
rows = sorted(rows, key=lambda r: r[0])
totals = {k: sum(amount for _, amount in g) for k, g in groupby(rows, key=___)}
check(12, totals)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `sort_employees`

Sort by team ascending, then salary descending.

In [ ]:
def sort_employees(rows: list[dict]) -> list[dict]:
    """Sort by 'team' ascending, then 'salary' descending. Hint: key=lambda r: (..., -...)."""
    return ___

In [ ]:
check('sort_employees')

### 2. `power`

Return `base` raised to `exponent`.

In [ ]:
def power(base: float, exponent: float) -> float:
    """Return base raised to exponent."""
    return ___

In [ ]:
check('power')

### 3. `make_square`

Build a square function from `power` with `functools.partial`.

In [ ]:
def make_square() -> Callable[[float], float]:
    """Return a one-argument square function built from power() with functools.partial."""
    return ___

In [ ]:
check('make_square')

### 4. `cached_fib`

Recursive Fibonacci memoised with `lru_cache`.

In [ ]:
def cached_fib(n: int) -> int:
    """Recursive Fibonacci memoized with functools.lru_cache (cached_fib(0) == 0)."""
    return ___

In [ ]:
check('cached_fib')

### 5. `all_variants`

Every (color, size) pair with `itertools.product`.

In [ ]:
def all_variants(colors: list[str], sizes: list[str]) -> list[tuple[str, str]]:
    """Every (color, size) pair, colors in the outer loop. Hint: itertools.product."""
    return ___

In [ ]:
check('all_variants')

### Independent build — `totals_by_category`

Order Desk receives a list of records with `'category'` (a string) and `'amount'` (a number). Return a new dictionary mapping every category to its total amount. Categories may appear out of order; amounts can be zero or negative (refunds). Preserve the records and their order. An empty list returns `{}`.

For `[{'category': 'paper', 'amount': 7}, {'category': 'pens', 'amount': 3}, {'category': 'paper', 'amount': -2}]`, return `{'paper': 5, 'pens': 3}`. Implement the interface and test an empty list.

**Delayed revision.** An adjustment arrives after another category's sale. Check that the original category's total changes without rewriting the input records. Explain which earlier course skill helps keep track of each category.


In [ ]:
def totals_by_category(rows: list[dict]) -> dict[str, float]:
    raise NotImplementedError

In [ ]:
check('totals_by_category')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** sorted(words, key=len) sorts by…

   - **a)** Reverse order
   - **b)** Alphabet
   - **c)** Word length

**2.** What must you do before itertools.groupby?

   - **a)** Convert to a set
   - **b)** Sort by the same key
   - **c)** Nothing

**3.** functools.lru_cache is safe on functions that…

   - **a)** Are pure: same input, same output
   - **b)** Depend on the current time
   - **c)** Print output

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/09', {1: '8a796902cf87', 2: 'c379b3b9e83b', 3: 'db9ae3e9baa4'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [10 — Typing, Contracts, and Protocols](10_type_hints_and_protocols.ipynb)